
## Folder 09 — Incremental Ingestion:


- We are moving from full-table processing to incremental processing. 
- The first step is to ingest only newly arriving CSV files into Bronze using Auto Loader.


**Objective:** 

Process new files without reprocessing files already tracked by the Auto Loader checkpoint.


**One limitation matters:**

- This handles new files, not updates or deletes inside previously processed files. 
- Your current source CSVs do not contain reliable CDC metadata, so we will not pretend this is full change-data capture.

In [0]:

# ============================================================
# E-COMMERCE LAKEHOUSE
# FOLDER 09: INCREMENTAL INGESTION
# FILE 01: INCREMENTAL BRONZE INGESTION
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

import uuid
import json


# ============================================================
# 2. ENVIRONMENT CONFIGURATION
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"

VOLUME_SCHEMA = "default"

VOLUME = "data_volume"

BASE_PATH = (
    f"/Volumes/{CATALOG}/{VOLUME_SCHEMA}/"
    f"{VOLUME}/ecommerce_lakehouse"
)

LANDING_PATH = f"{BASE_PATH}/landing"

CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints"

SCHEMA_PATH = f"{BASE_PATH}/schemas"

LOG_PATH = f"{BASE_PATH}/logs"


# ============================================================
# 3. DATASET CONFIGURATION
# ============================================================

DATASETS = [
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"
]


# ============================================================
# 4. EXECUTION CONFIGURATION
# ============================================================

# Customers has already been successfully ingested.
# Process only the remaining six datasets.

DATASETS_TO_PROCESS = [
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"
]

# Reuse the recovery checkpoint version.

CHECKPOINT_VERSION = "bronze_recovery_v2"


# ============================================================
# 5. CREATE REQUIRED SCHEMAS AND DIRECTORIES
# ============================================================

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {CATALOG}.{BRONZE_SCHEMA}
    """
)

dbutils.fs.mkdirs(CHECKPOINT_PATH)

dbutils.fs.mkdirs(SCHEMA_PATH)

dbutils.fs.mkdirs(LOG_PATH)

print("=" * 70)
print("ENVIRONMENT CONFIGURATION")
print("=" * 70)

print(f"Catalog: {CATALOG}")

print(f"Bronze Schema: {BRONZE_SCHEMA}")

print(f"Landing Path: {LANDING_PATH}")

print(f"Checkpoint Path: {CHECKPOINT_PATH}")

print(f"Recovery Version: {CHECKPOINT_VERSION}")

print(f"Datasets Selected: {DATASETS_TO_PROCESS}")


# ============================================================
# 6. STREAMING PROGRESS CONVERSION FUNCTION
# ============================================================

def convert_progress_to_dict(progress):

    if isinstance(progress, dict):
        return progress

    if isinstance(progress, str):

        return json.loads(progress)

    if hasattr(progress, "json"):

        progress_json = progress.json

        if callable(progress_json):
            progress_json = progress_json()

        return json.loads(progress_json)

    if hasattr(progress, "asDict"):

        return progress.asDict(recursive=True)

    raise TypeError(
        f"Unsupported streaming progress type: "
        f"{type(progress)}"
    )


# ============================================================
# 7. INCREMENTAL BRONZE INGESTION FUNCTION
# ============================================================

def incremental_bronze_ingestion(dataset_name):

    print("\n" + "=" * 70)

    print(
        f"STARTING INCREMENTAL INGESTION: {dataset_name}"
    )

    print("=" * 70)

    ingestion_run_id = str(uuid.uuid4())


    # --------------------------------------------------------
    # 7.1 CONFIGURATION
    # --------------------------------------------------------

    source_path = (
        f"{LANDING_PATH}/{dataset_name}"
    )

    target_table = (
        f"{CATALOG}.{BRONZE_SCHEMA}.{dataset_name}"
    )

    dataset_checkpoint = (
        f"{CHECKPOINT_PATH}/"
        f"{CHECKPOINT_VERSION}/{dataset_name}"
    )

    dataset_schema_path = (
        f"{SCHEMA_PATH}/bronze/{dataset_name}"
    )

    dbutils.fs.mkdirs(dataset_checkpoint)

    dbutils.fs.mkdirs(dataset_schema_path)

    print(f"Source: {source_path}")

    print(f"Target: {target_table}")

    print(f"Checkpoint: {dataset_checkpoint}")

    print(f"Schema Location: {dataset_schema_path}")

    print(f"Run ID: {ingestion_run_id}")


    # --------------------------------------------------------
    # 7.2 CHECK SOURCE DIRECTORY
    # --------------------------------------------------------

    try:

        source_files = dbutils.fs.ls(source_path)

    except Exception as error:

        raise RuntimeError(
            f"Source directory not accessible: "
            f"{source_path}. Error: {str(error)}"
        ) from error

    csv_files = [
        file.path
        for file in source_files
        if file.name.lower().endswith(".csv")
    ]

    if not csv_files:

        print(
            f"No CSV files found for {dataset_name}"
        )

        return {
            "dataset_name": dataset_name,
            "execution_status": "NO_SOURCE_FILES",
            "landing_records": 0,
            "processed_records": 0,
            "bronze_records": 0,
            "error_message": (
                "No CSV files found in landing directory."
            )
        }

    print(f"CSV Files Found: {len(csv_files)}")


    # --------------------------------------------------------
    # 7.3 CHECK LANDING RECORD COUNT
    # --------------------------------------------------------

    landing_df = (
        spark.read
        .option("header", "true")
        .option("multiLine", "true")
        .csv(csv_files)
    )

    landing_count = landing_df.count()

    print(f"Landing Records: {landing_count}")


    # --------------------------------------------------------
    # 7.4 CREATE AUTO LOADER STREAM
    # --------------------------------------------------------

    source_df = (

        spark.readStream

        .format("cloudFiles")

        .option(
            "cloudFiles.format",
            "csv"
        )

        .option(
            "cloudFiles.schemaLocation",
            dataset_schema_path
        )

        .option(
            "cloudFiles.inferColumnTypes",
            "true"
        )

        .option(
            "cloudFiles.schemaEvolutionMode",
            "addNewColumns"
        )

        .option(
            "cloudFiles.rescuedDataColumn",
            "_rescued_data"
        )

        .option(
            "header",
            "true"
        )

        .option(
            "multiLine",
            "true"
        )

        .option(
            "mode",
            "PERMISSIVE"
        )

        .load(source_path)
    )


    # --------------------------------------------------------
    # 7.5 ADD INGESTION METADATA
    # --------------------------------------------------------

    bronze_df = (

        source_df

        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )

        .withColumn(
            "_source_file_modification_time",
            F.col("_metadata.file_modification_time")
        )

        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )

        .withColumn(
            "_ingestion_date",
            F.current_date()
        )

        .withColumn(
            "_ingestion_run_id",
            F.lit(ingestion_run_id)
        )
    )


    # --------------------------------------------------------
    # 7.6 WRITE INTO BRONZE DELTA TABLE
    # --------------------------------------------------------

    streaming_query = (

        bronze_df

        .writeStream

        .format("delta")

        .outputMode("append")

        .option(
            "checkpointLocation",
            dataset_checkpoint
        )

        .option(
            "mergeSchema",
            "true"
        )

        .trigger(
            availableNow=True
        )

        .toTable(target_table)
    )

    streaming_query.awaitTermination()


    # --------------------------------------------------------
    # 7.7 CAPTURE STREAMING EXECUTION METRICS
    # --------------------------------------------------------

    recent_progress = streaming_query.recentProgress

    processed_records = 0

    print("\nSTREAMING PROGRESS DETAILS")

    print("-" * 60)

    print(
        f"Progress Events: {len(recent_progress)}"
    )

    for index, progress in enumerate(recent_progress):

        progress_dict = convert_progress_to_dict(
            progress
        )

        sources = progress_dict.get("sources", [])

        event_records = 0

        for source in sources:

            input_rows = source.get(
                "numInputRows",
                0
            )

            event_records += int(input_rows or 0)

        processed_records += event_records

        print(
            f"\nProgress Event: {index + 1}"
        )

        print(
            f"Batch ID: {progress_dict.get('batchId')}"
        )

        print(
            f"Records Processed: {event_records}"
        )

        print(
            f"Source Details: {sources}"
        )

    print(
        f"\nTotal Records Processed: {processed_records}"
    )


    # --------------------------------------------------------
    # 7.8 STREAMING QUERY DIAGNOSTICS
    # --------------------------------------------------------

    print("\nSTREAMING QUERY DIAGNOSTICS")

    print("-" * 60)

    print(
        f"Query Active: {streaming_query.isActive}"
    )

    print(
        f"Query Status: {streaming_query.status}"
    )

    print(
        f"Query Exception: {streaming_query.exception()}"
    )


    # --------------------------------------------------------
    # 7.9 VALIDATE BRONZE TABLE
    # --------------------------------------------------------

    if spark.catalog.tableExists(target_table):

        total_bronze_records = (
            spark.table(target_table).count()
        )

    else:

        total_bronze_records = 0

    print("\nINGESTION METRICS")

    print("-" * 60)

    print(f"Landing Records: {landing_count}")

    print(f"Processed Records: {processed_records}")

    print(
        f"Total Bronze Records: {total_bronze_records}"
    )

    print(f"CSV Files Found: {len(csv_files)}")


    # --------------------------------------------------------
    # 7.10 DETERMINE EXECUTION STATUS
    # --------------------------------------------------------

    if (
        processed_records > 0
        and total_bronze_records > 0
    ):

        execution_status = "SUCCESS"

        error_message = None

    elif total_bronze_records > 0:

        execution_status = "NO_NEW_DATA"

        error_message = (
            "No new records processed during this run. "
            "Existing Bronze records are available."
        )

    elif landing_count == 0:

        execution_status = "NO_NEW_DATA"

        error_message = (
            "Landing CSV files contain no data records."
        )

    else:

        execution_status = "FAILED"

        error_message = (
            "Landing files contain records, but Bronze "
            "ingestion returned zero records. "
            "Inspect streaming diagnostics."
        )

    print(
        f"\nExecution Status: {execution_status}"
    )

    if error_message:

        print(f"Details: {error_message}")

    print("=" * 70)

    return {
        "dataset_name": dataset_name,
        "execution_status": execution_status,
        "landing_records": landing_count,
        "processed_records": processed_records,
        "bronze_records": total_bronze_records,
        "error_message": error_message
    }


# ============================================================
# 8. PROCESS SELECTED DATASETS
# ============================================================

ingestion_results = []

for dataset in DATASETS_TO_PROCESS:

    try:

        result = incremental_bronze_ingestion(dataset)

        ingestion_results.append(result)

    except Exception as error:

        error_message = str(error)[:1000]

        print(
            f"\nERROR PROCESSING {dataset}: "
            f"{error_message}"
        )

        ingestion_results.append(
            {
                "dataset_name": dataset,
                "execution_status": "FAILED",
                "landing_records": 0,
                "processed_records": 0,
                "bronze_records": 0,
                "error_message": error_message
            }
        )


# ============================================================
# 9. CREATE INGESTION EXECUTION REPORT
# ============================================================

result_schema = StructType([

    StructField(
        "dataset_name",
        StringType(),
        True
    ),

    StructField(
        "execution_status",
        StringType(),
        True
    ),

    StructField(
        "landing_records",
        StringType(),
        True
    ),

    StructField(
        "processed_records",
        StringType(),
        True
    ),

    StructField(
        "bronze_records",
        StringType(),
        True
    ),

    StructField(
        "error_message",
        StringType(),
        True
    )

])

result_rows = [

    (
        result["dataset_name"],
        result["execution_status"],
        str(result["landing_records"]),
        str(result["processed_records"]),
        str(result["bronze_records"]),
        result["error_message"]
    )

    for result in ingestion_results

]

result_df = spark.createDataFrame(
    result_rows,
    schema=result_schema
)

result_df = result_df.withColumn(
    "execution_timestamp",
    F.current_timestamp()
)

result_table = (
    f"{CATALOG}.{BRONZE_SCHEMA}."
    "incremental_ingestion_report"
)

(
    result_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(result_table)
)

print("\nIngestion execution report saved.")


# ============================================================
# 10. DISPLAY EXECUTION REPORT
# ============================================================

display(

    spark.table(result_table)

    .orderBy(
        F.col("execution_timestamp").desc()
    )

)


# ============================================================
# 11. FINAL BRONZE VALIDATION
# ============================================================

print("\n" + "=" * 70)

print("FINAL BRONZE TABLE VALIDATION")

print("=" * 70)

total_all_bronze_records = 0

for dataset in DATASETS:

    target_table = (
        f"{CATALOG}.{BRONZE_SCHEMA}.{dataset}"
    )

    if spark.catalog.tableExists(target_table):

        bronze_count = (
            spark.table(target_table).count()
        )

        total_all_bronze_records += bronze_count

        print(
            f"{dataset}: {bronze_count} records"
        )

    else:

        print(
            f"{dataset}: Bronze table does not exist."
        )

print("-" * 70)

print(
    f"TOTAL BRONZE RECORDS: {total_all_bronze_records}"
)


# ============================================================
# 12. INGESTION SUMMARY
# ============================================================

success_count = sum(
    1 for result in ingestion_results
    if result["execution_status"] == "SUCCESS"
)

no_new_data_count = sum(
    1 for result in ingestion_results
    if result["execution_status"] in [
        "NO_NEW_DATA",
        "NO_SOURCE_FILES"
    ]
)

failed_count = sum(
    1 for result in ingestion_results
    if result["execution_status"] == "FAILED"
)

print("\n" + "=" * 70)

print("INCREMENTAL INGESTION SUMMARY")

print("=" * 70)

print(
    f"Total Datasets Selected: {len(DATASETS_TO_PROCESS)}"
)

print(f"Successful: {success_count}")

print(
    f"No New Data / No Source Files: {no_new_data_count}"
)

print(f"Failed: {failed_count}")

print(
    f"Total Records Across All Bronze Tables: "
    f"{total_all_bronze_records}"
)

print("=" * 70)

if failed_count > 0:

    print(
        "WARNING: Some datasets failed. "
        "Investigate the execution report."
    )

elif success_count == len(DATASETS_TO_PROCESS):

    print(
        "All selected datasets were successfully ingested."
    )

else:

    print(
        "Execution completed. Review dataset statuses."
    )



### Validate all seven Bronze tables

In [0]:
%sql
SELECT 'customers' AS dataset, COUNT(*) AS total_records
FROM workspace.ecommerce_bronze.customers

UNION ALL

SELECT 'products', COUNT(*)
FROM workspace.ecommerce_bronze.products

UNION ALL

SELECT 'orders', COUNT(*)
FROM workspace.ecommerce_bronze.orders

UNION ALL

SELECT 'order_items', COUNT(*)
FROM workspace.ecommerce_bronze.order_items

UNION ALL

SELECT 'payments', COUNT(*)
FROM workspace.ecommerce_bronze.payments

UNION ALL

SELECT 'shipments', COUNT(*)
FROM workspace.ecommerce_bronze.shipments

UNION ALL

SELECT 'inventory_snapshots', COUNT(*)
FROM workspace.ecommerce_bronze.inventory_snapshots;


## 2. How Incremental Ingestion Works


### Incremental Ingestion Architecture:


```text
┌──────────────────────────────────────────────┐
│                 LANDING FOLDER               │
│                                              │
│          New CSV Files Arrive                │
└──────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│                  AUTO LOADER                  │
│                                              │
│       Discovers Files and Reads New Data     │
└──────────────────────────────────────────────┘
                       │
          ┌────────────┴────────────┐
          ▼                         ▼
┌──────────────────────┐  ┌──────────────────────┐
│      CHECKPOINT      │  │    SCHEMA LOCATION   │
│                      │  │                      │
│ Tracks Processed     │  │ Tracks Inferred      │
│ Files and Progress   │  │ Schema Information  │
└──────────────────────┘  └──────────────────────┘
          │                         │
          └────────────┬────────────┘
                       ▼
┌──────────────────────────────────────────────┐
│              BRONZE DELTA TABLES             │
│                                              │
│          Newly Ingested Records              │
│                                              │
│          New Records Appended                │
└──────────────────────────────────────────────┘
```


### Key Components


| Component | Responsibility |
|---|---|
| Landing Folder | Receives incoming CSV files. |
| Auto Loader | Discovers and processes new files. |
| Checkpoint | Tracks streaming progress and processed files. |
| Schema Location | Stores inferred schema information and schema evolution history. |
| Bronze Delta Tables | Store newly ingested raw records. |


## 3. How to Test Incremental Ingestion


**Important:** 

- Do not test incremental ingestion by simply rerunning the notebook with the original CSV files. 
- That only demonstrates that the existing checkpoint prevents already processed files from being processed again.


### Step 1: Initial Ingestion:


Run the Bronze ingestion notebook once to process all existing CSV files.

**Expected result:**

- Existing files are discovered.
- Records are ingested into Bronze Delta tables.
- Checkpoint information is maintained.


### Step 2: Create a New CSV File


Create a small CSV file named:

`customers_incremental_001.csv`

Use the same customer schema:


```csv
customer_id,first_name,last_name,email,phone,city,state,created_at

CUST90001,Test,Customer,test90001@example.com,9876543210,Chennai,Tamil Nadu,2026-11-06
```


### Step 3: Upload the New File


Upload the CSV file into the following landing directory:


```text
/Volumes/workspace/default/data_volume/ecommerce_lakehouse/landing/customers/
```


### Step 4: Run the Notebook Again


Execute the Bronze ingestion notebook.

**Expected behavior:**

- Auto Loader discovers the newly uploaded file.
- The checkpoint identifies previously processed files.
- Only the new file is processed.
- The new record is appended to the Bronze Delta table.


### Step 5: Verify the Newly Ingested Record



Execute:

```sql
SELECT *
FROM workspace.ecommerce_bronze.customers
WHERE customer_id = 'CUST90001';
```

**Expected Result:**

| customer_id | first_name | last_name | city |
|---|---|---|---|
| CUST90001 | Test | Customer | Chennai |

Expected record count: **1**


### Step 6: Test Duplicate Prevention


Run the same notebook again without uploading another CSV file.

Execute:

```sql
SELECT COUNT(*) AS record_count
FROM workspace.ecommerce_bronze.customers
WHERE customer_id = 'CUST90001';
```


**Expected result:**


| Test | Expected Count |
|---|---:|
| After first ingestion | 1 |
| After second notebook execution | 1 |

The same file should not be ingested again, provided the checkpoint is unchanged and the Auto Loader processing configuration remains consistent.


## 4. Expected Test Results


| Test Scenario | Expected Behavior |
|---|---|
| Initial ingestion | All existing files processed. |
| New CSV uploaded | Auto Loader discovers the new file. |
| Second execution | Only newly discovered files are processed. |
| Same file processed again | Previously processed file is skipped. |
| Checkpoint retained | Previous processing progress is preserved. |


**Conclusion:** 

Incremental ingestion allows the E-Commerce Lakehouse to process newly arriving CSV files without repeatedly processing previously ingested files.


**Important interview point:** 

- Auto Loader provides file-level incremental discovery, not automatic business-key deduplication. 
- If a new file contains a customer ID that already exists in Bronze, the record can still be appended. 
- Preventing duplicate business records requires separate deduplication or MERGE logic.